In [38]:
import openai
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PayloadSchemaType, PointStruct, MatchAny, FieldCondition, Filter, Prefetch, FusionQuery

import pandas as pd
import numpy as np
import openai
import json
import tiktoken



In [2]:
qdrant_client = QdrantClient(url = "http://localhost:6333")

In [3]:
dummy_vector = np.zeros(1536).tolist()

In [4]:
payload = qdrant_client.query_points(
    collection_name="Amazon-items-collection-02-hybrid-search",
    query= dummy_vector,
    using = "text-embedding-3-small",
    limit = 1000,
    with_payload = ["parent_asin"],
    with_vectors = False
)

In [7]:
len(payload.points)

1000

In [5]:
parent_asin_list = [item.payload['parent_asin'] for item in payload.points]

In [6]:
parent_asin_list

['B0BMVZXJYM',
 'B0BDQSYQKG',
 'B0BN8Q1Z9S',
 'B0B2NVGBN2',
 'B0B9FTVL58',
 'B0BZSGNYXM',
 'B0B6ZZH83Y',
 'B0BP2CR2RS',
 'B09WVHZ4X8',
 'B0B3J7V1V6',
 'B0BQZ23TJL',
 'B09W5RJ1LY',
 'B09QPH2GN3',
 'B09Q8FLNDR',
 'B09XQ3H6Y4',
 'B09QGNB537',
 'B0BD4RWZLZ',
 'B0C64P1WX4',
 'B09QPQ9JRF',
 'B0B1F4FKPF',
 'B097GDDN1X',
 'B09DKGPST6',
 'B09XN72MP6',
 'B0BF9DQB6K',
 'B0BFDLTRGX',
 'B0B46D4FM3',
 'B0B2W9G1M8',
 'B0B2V9P3K4',
 'B09SL5RQ6L',
 'B098K6N6TX',
 'B09Q3DH7V3',
 'B09SYSRBLC',
 'B0BBG6P5SM',
 'B0B965NYT9',
 'B0BSF9NZ3V',
 'B09TTBWHN5',
 'B09T5YML6T',
 'B0BL2HJ8WS',
 'B0B5H7T7XZ',
 'B0B9ZX2DCJ',
 'B09PZ8W2ZG',
 'B0BS3V1QBY',
 'B0BLJXPRL9',
 'B09R1TMXR3',
 'B09PN99MN9',
 'B09TP3SZ7C',
 'B0C2YMY2QB',
 'B0C7KJJ46S',
 'B09T7YT8HC',
 'B0BCCCNHD8',
 'B0B8ZHF99W',
 'B09TPPL2Q1',
 'B0BW89VYWN',
 'B0CHN3BSQM',
 'B0BP1YXF3R',
 'B0B71N3CTS',
 'B0C85F21JR',
 'B0BYYGZHG5',
 'B0B68DHY1J',
 'B0B31H2TBX',
 'B0BVGRBDRH',
 'B0BL12DTJ5',
 'B0B45VPMT4',
 'B09PQ97H3G',
 'B09PY7NCSL',
 'B0B4N1D73R',
 'B0BLDZZD

In [7]:
df_reviews = pd.read_json("../../data/Electornics_2022_2023_with_category_ratings_100_sample_1000.jsonl", lines = True)

In [8]:
df_reviews.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5,Perfect!,This is perfect! Thank you so much!!! I absolu...,[],B09992M2LX,B09ZPV8WBV,AHX4XWVVQUKT3FCNWCVASDF4Q56Q,2022-08-05 04:06:39.589,0,True
1,5,3ft mini usb cables,I don't have many things that still use a mini...,[],B09Y94B2NM,B09Y95BMKX,AFZUK3MTBIBEDQOPAK3OATUOUKLA,2022-07-16 16:03:28.714,3,True
2,5,I would buy it again.,Great product. Worked well for what we needed ...,[],B07T55DL33,B0B2JWCMCY,AF5KFHNT3TQJ2GNSE3FCDFQOBICA,2019-12-09 22:35:00.531,0,True
3,5,Great to Have Around,My husband and I were recently working a booth...,[],B09M89JN7B,B0BYYGZHG5,AHV6QCNBJNSGLATP56JAWJ3C4G2A,2022-03-22 01:43:49.342,0,False
4,5,Easy to use,Work as advertised and at a very good price.,[],B07T55DL33,B0B2JWCMCY,AG7WKTZINOFIXMZJYIPKIB7PV7NQ,2019-12-28 06:12:24.960,0,True


In [9]:
df_reviews_sample = df_reviews[df_reviews["parent_asin"].isin(parent_asin_list)]

In [10]:
len(df_reviews_sample)

105918

In [11]:
df_reviews_sample.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5,Perfect!,This is perfect! Thank you so much!!! I absolu...,[],B09992M2LX,B09ZPV8WBV,AHX4XWVVQUKT3FCNWCVASDF4Q56Q,2022-08-05 04:06:39.589,0,True
1,5,3ft mini usb cables,I don't have many things that still use a mini...,[],B09Y94B2NM,B09Y95BMKX,AFZUK3MTBIBEDQOPAK3OATUOUKLA,2022-07-16 16:03:28.714,3,True
2,5,I would buy it again.,Great product. Worked well for what we needed ...,[],B07T55DL33,B0B2JWCMCY,AF5KFHNT3TQJ2GNSE3FCDFQOBICA,2019-12-09 22:35:00.531,0,True
3,5,Great to Have Around,My husband and I were recently working a booth...,[],B09M89JN7B,B0BYYGZHG5,AHV6QCNBJNSGLATP56JAWJ3C4G2A,2022-03-22 01:43:49.342,0,False
4,5,Easy to use,Work as advertised and at a very good price.,[],B07T55DL33,B0B2JWCMCY,AG7WKTZINOFIXMZJYIPKIB7PV7NQ,2019-12-28 06:12:24.960,0,True


#### Define functions to preprocess reviews data

In [12]:
def preprocess_review_data(row):
    return f"{row['title']} {row['text']}"

In [13]:
encoding = tiktoken.encoding_for_model("text-embedding-3-small")

In [14]:
encoding.encode("i really like my strawberries")

[72, 2216, 1093, 856, 76203]

In [15]:
def token_count(row, model= "text-embedding-3-small"):
    encoding = tiktoken.encoding_for_model(model)
    return len(encoding.encode(row['preprocessed_data']))

In [16]:
df_reviews_sample['preprocessed_data'] = df_reviews_sample.apply(preprocess_review_data, axis=1)

In [17]:
df_reviews_sample['preprocessed_data_token_count'] = df_reviews_sample.apply(token_count, axis=1)

In [27]:
df_reviews_sample.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,preprocessed_data,preprocessed_data_token_count
0,5,Perfect!,This is perfect! Thank you so much!!! I absolu...,[],B09992M2LX,B09ZPV8WBV,AHX4XWVVQUKT3FCNWCVASDF4Q56Q,2022-08-05 04:06:39.589,0,True,Perfect! This is perfect! Thank you so much!!!...,21
1,5,3ft mini usb cables,I don't have many things that still use a mini...,[],B09Y94B2NM,B09Y95BMKX,AFZUK3MTBIBEDQOPAK3OATUOUKLA,2022-07-16 16:03:28.714,3,True,3ft mini usb cables I don't have many things t...,114
2,5,I would buy it again.,Great product. Worked well for what we needed ...,[],B07T55DL33,B0B2JWCMCY,AF5KFHNT3TQJ2GNSE3FCDFQOBICA,2019-12-09 22:35:00.531,0,True,I would buy it again. Great product. Worked we...,24
3,5,Great to Have Around,My husband and I were recently working a booth...,[],B09M89JN7B,B0BYYGZHG5,AHV6QCNBJNSGLATP56JAWJ3C4G2A,2022-03-22 01:43:49.342,0,False,Great to Have Around My husband and I were rec...,76
4,5,Easy to use,Work as advertised and at a very good price.,[],B07T55DL33,B0B2JWCMCY,AG7WKTZINOFIXMZJYIPKIB7PV7NQ,2019-12-28 06:12:24.960,0,True,Easy to use Work as advertised and at a very g...,13


In [26]:
len(df_reviews_sample)

105918

#### Qdrant Create Collection for reviews

In [20]:
qdrant_client.create_collection(
    collection_name = "Amazon-items-collection-01-reviews",
    vectors_config = VectorParams(size = 1536, distance = Distance.COSINE)
)

True

In [21]:
qdrant_client.create_payload_index(
    collection_name = "Amazon-items-collection-01-reviews",
    field_name= "parent_asin",
    field_schema= PayloadSchemaType.KEYWORD
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

#### Embedding Functions

In [48]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input = [text],
        model = model
    )
    return response.data[0].embedding

In [40]:
def get_embeddings_batch(text_list, model = "text-embedding-3-small", batch_size = 100):
    if len(text_list) < batch_size:
        response = openai.embeddings.create(input=text_list, model = model)
        return [embedding.embedding for embedding in response.data]
    
    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i+ batch_size]
        response = openai.embeddings.create(input= batch, model=model)
        all_embeddings.extend([embedding.embedding for embedding in response.data])
        print(f"Processed {counter *batch_size} of {len(text_list)}")
        counter +=1
    return all_embeddings

#### Embed the text and add aditional fields to the payload of each vector for reviews

In [29]:
data_to_embed_reviews = df_reviews_sample[["preprocessed_data", "parent_asin"]].to_dict(orient="records")

In [30]:
data_to_embed_reviews

[{'preprocessed_data': 'Perfect! This is perfect! Thank you so much!!! I absolutely love it!!! It’s great quality!!!',
  'parent_asin': 'B09ZPV8WBV'},
 {'preprocessed_data': "3ft mini usb cables I don't have many things that still use a mini USB charger cable, but I after a while you have to replace the charger cables.  These seem to work well and it was noted in the ad that they are reinforced at the head area to prolong life.  We shall see - time will tell.  I never hesitate to update my reviews should new info seem useful.  I do not accept any discounts or deals that are not available to all shoppers. And my reviews are based purely on my personal experience with each item I review.",
  'parent_asin': 'B09Y95BMKX'},
 {'preprocessed_data': 'I would buy it again. Great product. Worked well for what we needed it for. Would buy it again.',
  'parent_asin': 'B0B2JWCMCY'},
 {'preprocessed_data': 'Great to Have Around My husband and I were recently working a booth at a trade show for his b

In [31]:
text_to_embed_reviews = [data["preprocessed_data"] for data in data_to_embed_reviews]

In [32]:
text_to_embed_reviews

['Perfect! This is perfect! Thank you so much!!! I absolutely love it!!! It’s great quality!!!',
 "3ft mini usb cables I don't have many things that still use a mini USB charger cable, but I after a while you have to replace the charger cables.  These seem to work well and it was noted in the ad that they are reinforced at the head area to prolong life.  We shall see - time will tell.  I never hesitate to update my reviews should new info seem useful.  I do not accept any discounts or deals that are not available to all shoppers. And my reviews are based purely on my personal experience with each item I review.",
 'I would buy it again. Great product. Worked well for what we needed it for. Would buy it again.',
 'Great to Have Around My husband and I were recently working a booth at a trade show for his business. I set up a charging station with these. It was perfect! We were able to accommodate everyone who needed to charge their phone! The cables are a good length, not too short or t

In [41]:
embedding_reviews = get_embeddings_batch(text_to_embed_reviews, batch_size=500)

Processed 500 of 105918
Processed 1000 of 105918
Processed 1500 of 105918
Processed 2000 of 105918
Processed 2500 of 105918
Processed 3000 of 105918
Processed 3500 of 105918
Processed 4000 of 105918
Processed 4500 of 105918
Processed 5000 of 105918
Processed 5500 of 105918
Processed 6000 of 105918
Processed 6500 of 105918
Processed 7000 of 105918
Processed 7500 of 105918
Processed 8000 of 105918
Processed 8500 of 105918
Processed 9000 of 105918
Processed 9500 of 105918
Processed 10000 of 105918
Processed 10500 of 105918
Processed 11000 of 105918
Processed 11500 of 105918
Processed 12000 of 105918
Processed 12500 of 105918
Processed 13000 of 105918
Processed 13500 of 105918
Processed 14000 of 105918
Processed 14500 of 105918
Processed 15000 of 105918
Processed 15500 of 105918
Processed 16000 of 105918
Processed 16500 of 105918
Processed 17000 of 105918
Processed 17500 of 105918
Processed 18000 of 105918
Processed 18500 of 105918
Processed 19000 of 105918
Processed 19500 of 105918
Proces

In [42]:
len(embedding_reviews)

105918

In [44]:
pointstructs = []
i = 1
for embedding, data in zip(embedding_reviews, data_to_embed_reviews):
    pointstructs.append(
        PointStruct(
            id=i,
            vector=embedding,
            payload= {
                "text": data["preprocessed_data"],
                "parent_asin": data["parent_asin"],
            }
        )
    )
    i += 1

In [45]:
batch_size_qdrant =100
counter =1
for i in range(0, len(pointstructs), batch_size_qdrant):
    batch = pointstructs[i:i+batch_size_qdrant]
    qdrant_client.upsert(
        collection_name="Amazon-items-collection-01-reviews",
        wait=True,
        points=batch
    )
    print(f"Processed {counter +batch_size_qdrant} of {len(pointstructs)}")
    counter +=1

Processed 101 of 105918
Processed 102 of 105918
Processed 103 of 105918
Processed 104 of 105918
Processed 105 of 105918
Processed 106 of 105918
Processed 107 of 105918
Processed 108 of 105918
Processed 109 of 105918
Processed 110 of 105918
Processed 111 of 105918
Processed 112 of 105918
Processed 113 of 105918
Processed 114 of 105918
Processed 115 of 105918
Processed 116 of 105918
Processed 117 of 105918
Processed 118 of 105918
Processed 119 of 105918
Processed 120 of 105918
Processed 121 of 105918
Processed 122 of 105918
Processed 123 of 105918
Processed 124 of 105918
Processed 125 of 105918
Processed 126 of 105918
Processed 127 of 105918
Processed 128 of 105918
Processed 129 of 105918
Processed 130 of 105918
Processed 131 of 105918
Processed 132 of 105918
Processed 133 of 105918
Processed 134 of 105918
Processed 135 of 105918
Processed 136 of 105918
Processed 137 of 105918
Processed 138 of 105918
Processed 139 of 105918
Processed 140 of 105918
Processed 141 of 105918
Processed 142 of

#### function to run search against reviews of prefiltered set of products IDs

In [49]:
def retrieve_prefiltered_reviews_data(query, parent_asins, k=5):
    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name = "Amazon-items-collection-01-reviews",
        prefetch=[
            Prefetch(
                query=query_embedding,
                filter=Filter(
                    must=[
                        FieldCondition(
                            key="parent_asin",
                            match=MatchAny(
                                any=parent_asins
                            )
                        )
                    ]
                ),
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    )
    return results

In [50]:
reviews = retrieve_prefiltered_reviews_data("bad quality", ["B0B9GHF8QM"], k=5)

In [51]:
reviews.points

[ScoredPoint(id=49044, version=493, score=0.5, payload={'text': 'Ok Quality The sound is very low', 'parent_asin': 'B0B9GHF8QM'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=101072, version=1013, score=0.33333334, payload={'text': 'Quality great Great quality', 'parent_asin': 'B0B9GHF8QM'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=49074, version=493, score=0.25, payload={'text': 'Not good they did not match up to the quality of the way they looked in the picture.', 'parent_asin': 'B0B9GHF8QM'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=38074, version=383, score=0.2, payload={'text': 'Sound Sound is not good. I expected more, i understand is cheap but still the sound is not good.', 'parent_asin': 'B0B9GHF8QM'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=48905, version=492, score=0.16666667, payload={'text': "awful sound quality Never imagined that I will find a earbud with such a bad sound quality in 2

In [52]:
reviews = retrieve_prefiltered_reviews_data("bad quality", ["B0B9GHF8QM", "B0BGRL2618"], k=5)

In [53]:
reviews.points

[ScoredPoint(id=63043, version=633, score=0.5, payload={'text': "Sounds cheap Sound quality isn't good at high volumes", 'parent_asin': 'B0BGRL2618'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=49074, version=493, score=0.33333334, payload={'text': 'Not good they did not match up to the quality of the way they looked in the picture.', 'parent_asin': 'B0B9GHF8QM'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=30978, version=312, score=0.25, payload={'text': 'Not the best quality These were ok but I kept having a hard time syncing them to my phone not sure why. Also the sound quality was not the best maybe I got some defective ones but they weren’t very good. I know these aren’t the highest quality but I was extra bit more from them.', 'parent_asin': 'B0BGRL2618'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=62948, version=632, score=0.2, payload={'text': 'Over all quality I would not buy theses again or recommend them to someon